<a href="https://colab.research.google.com/github/eduloopezzz/TFG_ANALISISDATO_EDUARDOLOPEZ/blob/main/TFGBAANALISIS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
================================================================================
PILAR 2 — MODELO A: EXPLICATIVO (Regresión Lineal sin lag)
================================================================================
TFG: Criminalidad patrimonial y factores socioeconómicos en España (2010-2024)
Grado en Business Analytics — UFV 2025-26

OBJETIVO DE ESTE MODELO:
  Responder a la pregunta EXPLICATIVA: ¿Qué factores socioeconómicos explican
  la distribución del crimen entre provincias y su evolución temporal?

  Al NO incluir el lag-1 de la variable objetivo, las variables económicas
  (paro, renta, densidad) compiten en igualdad de condiciones y sus coeficientes
  son interpretables como efectos parciales sobre la criminalidad.

VARIABLES:
  Y  : Tasa_Delitos_100k (delitos patrimoniales por 100.000 habitantes)
  X1 : Tasa_Paro
  X2 : Renta_Neta_Media
  X3 : Densidad_Poblacion
  X4 : Dummy_COVID (1 = años 2020-2021)

FIGURAS GENERADAS (carpeta ./figuras_modeloA/):
  fig_A_01_tabla_metricas.png   — Métricas train/test + CV
  fig_A_02_scatter.png          — Real vs. Predicho en test
  fig_A_03_coeficientes.png     — Coeficientes β estandarizados
  fig_A_04_evolucion.png        — Evolución temporal real vs. predicho
  fig_A_05_residuos.png         — Distribución de residuos en test
  fig_A_06_cv_folds.png         — Walk-Forward CV por fold
  fig_A_07_pdp.png              — Dependencia Parcial de cada predictor
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings, os
warnings.filterwarnings('ignore')

# ── Estilo global ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})
C = {
    'azul':     '#2E75B6',
    'azul_osc': '#1F497D',
    'gris':     '#595959',
    'naranja':  '#E36C09',
    'verde':    '#375623',
    'rojo':     '#C0392B',
    'claro':    '#BDD7EE',
}

OUT = './figuras_modeloA'
os.makedirs(OUT, exist_ok=True)

# ==============================================================================
# 1. CARGA DE DATOS
# ==============================================================================
print("=" * 72)
print("  MODELO A — EXPLICATIVO (Regresión Lineal sin lag-1)")
print("=" * 72)

master = pd.read_csv('base_datos_maestra.csv')
print(f"\nDataset: {master.shape[0]} filas × {master.shape[1]} columnas")
print(f"Provincias: {master['Provincia'].nunique()} | Años: {master['Año'].min()}-{master['Año'].max()}")

Y_COL  = 'Tasa_Delitos_100k'
X_COLS = ['Tasa_Paro', 'Renta_Neta_Media', 'Densidad_Poblacion', 'Dummy_COVID']
LABEL  = {
    'Tasa_Paro':          'Tasa de Paro (X₁)',
    'Renta_Neta_Media':   'Renta Neta Media (X₂)',
    'Densidad_Poblacion': 'Densidad Poblacional (X₃)',
    'Dummy_COVID':        'Dummy COVID (X₄)',
}

# NOTA: No se aplica feature engineering temporal.
# El dataset se usa completo (2010-2024) porque no necesitamos calcular lag-1
# y por tanto no perdemos el año 2010.
print(f"\nNOTA: Modelo explicativo puro — sin lag-1, sin delta paro.")
print(f"Dataset completo: {len(master)} observaciones ({master['Año'].min()}-{master['Año'].max()})")

# ==============================================================================
# 2. PARTICIÓN TEMPORAL TRAIN / TEST
# ==============================================================================
YEAR_SPLIT = 2022

train = master[master['Año'] <  YEAR_SPLIT].copy()
test  = master[master['Año'] >= YEAR_SPLIT].copy()

X_train = train[X_COLS].values
y_train = train[Y_COL].values
X_test  = test[X_COLS].values
y_test  = test[Y_COL].values

print(f"\nPartición temporal:")
print(f"  Train: {len(train)} obs  (años {train['Año'].min()}-{train['Año'].max()})")
print(f"  Test:  {len(test)} obs  (años {test['Año'].min()}-{test['Año'].max()})")

# Escalado estándar (media 0, desviación 1) — necesario para comparar coeficientes
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ==============================================================================
# 3. VALIDACIÓN WALK-FORWARD
# ==============================================================================
# En datos de panel temporal, el KFold estándar mezcla pasado y futuro.
# Walk-Forward entrena con todos los años ≤ t y valida con t+1,
# respetando la estructura causal del tiempo.

años_train  = sorted(train['Año'].unique())
MIN_AÑOS    = 4   # mínimo de años para entrenar antes de validar

wf_splits = []
for val_year in años_train:
    train_years = [y for y in años_train if y < val_year]
    if len(train_years) < MIN_AÑOS:
        continue
    tr_idx = train[train['Año'].isin(train_years)].index.values
    va_idx = train[train['Año'] == val_year].index.values
    wf_splits.append((tr_idx, va_idx))

print(f"\nWalk-Forward CV: {len(wf_splits)} folds")
for i, (tr_idx, va_idx) in enumerate(wf_splits):
    tr_y = sorted(train.loc[tr_idx, 'Año'].unique())
    va_y = train.loc[va_idx, 'Año'].unique()[0]
    print(f"  Fold {i+1}: train {tr_y[0]}-{tr_y[-1]} → val {va_y}")

# ==============================================================================
# 4. ENTRENAMIENTO Y EVALUACIÓN
# ==============================================================================
print(f"\n{'─'*72}")
print("  ENTRENAMIENTO")
print(f"{'─'*72}")

lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_train = lr.predict(X_train_sc)
y_pred_test  = lr.predict(X_test_sc)

# Coeficientes
print("\nCoeficientes estandarizados (β):")
for col, b in zip(X_COLS, lr.coef_):
    direccion = "↑ más delitos" if b > 0 else "↓ menos delitos"
    print(f"  {LABEL[col]:30s}  β = {b:+8.2f}  ({direccion})")
print(f"  {'Intercepto':30s}  β₀= {lr.intercept_:8.2f}")

# Walk-Forward CV
def calc_metricas(y_r, y_p):
    mask = y_r != 0
    return {
        'RMSE': np.sqrt(mean_squared_error(y_r, y_p)),
        'MAE':  mean_absolute_error(y_r, y_p),
        'R2':   r2_score(y_r, y_p),
        'MAPE': np.mean(np.abs((y_r[mask] - y_p[mask]) / y_r[mask])) * 100,
    }

cv_r2, cv_rmse = [], []
for tr_idx, va_idx in wf_splits:
    tr_pos = np.where(np.isin(train.index.values, tr_idx))[0]
    va_pos = np.where(np.isin(train.index.values, va_idx))[0]
    sc_fold = StandardScaler()
    Xtr = sc_fold.fit_transform(X_train[tr_pos])
    Xva = sc_fold.transform(X_train[va_pos])
    m   = LinearRegression().fit(Xtr, y_train[tr_pos])
    yp  = m.predict(Xva)
    cv_r2.append(r2_score(y_train[va_pos], yp))
    cv_rmse.append(np.sqrt(mean_squared_error(y_train[va_pos], yp)))

cv_r2   = np.array(cv_r2)
cv_rmse = np.array(cv_rmse)
print(f"\nWalk-Forward CV ({len(wf_splits)} folds):")
print(f"  R²:   {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"  RMSE: {cv_rmse.mean():.1f} ± {cv_rmse.std():.1f}")

# Métricas finales
m_tr = calc_metricas(y_train, y_pred_train)
m_te = calc_metricas(y_test,  y_pred_test)

print(f"\nMétricas finales:")
print(f"  {'':20s} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'MAPE':>8}")
print(f"  {'Train':20s} {m_tr['RMSE']:>8.1f} {m_tr['MAE']:>8.1f} {m_tr['R2']:>8.4f} {m_tr['MAPE']:>7.1f}%")
print(f"  {'Test':20s}  {m_te['RMSE']:>8.1f} {m_te['MAE']:>8.1f} {m_te['R2']:>8.4f} {m_te['MAPE']:>7.1f}%")

# ==============================================================================
# 5. FIGURAS
# ==============================================================================
print(f"\n{'─'*72}")
print("  GENERANDO FIGURAS")
print(f"{'─'*72}")

# ── Fig A-01: Tabla de métricas ───────────────────────────────────────────────
print("  Fig A-01 — Tabla métricas")
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.axis('off')
header = ['Modelo', 'RMSE\nTrain', 'RMSE\nTest', 'MAE\nTrain', 'MAE\nTest',
          'R²\nTrain', 'R²\nTest', 'MAPE\nTrain', 'MAPE\nTest', 'WF-CV\nR² (μ±σ)']
row = [
    'Modelo A\n(Explicativo)',
    f"{m_tr['RMSE']:.1f}", f"{m_te['RMSE']:.1f}",
    f"{m_tr['MAE']:.1f}",  f"{m_te['MAE']:.1f}",
    f"{m_tr['R2']:.4f}",   f"{m_te['R2']:.4f}",
    f"{m_tr['MAPE']:.1f}%", f"{m_te['MAPE']:.1f}%",
    f"{cv_r2.mean():.3f}±{cv_r2.std():.3f}",
]
t = ax.table(cellText=[row], colLabels=header, cellLoc='center',
             loc='center', bbox=[0, 0.1, 1, 0.85])
t.auto_set_font_size(False)
t.set_fontsize(10)
for j in range(len(header)):
    t[0, j].set_facecolor(C['azul'])
    t[0, j].set_text_props(color='white', fontweight='bold', fontsize=9)
    t[1, j].set_facecolor('#EBF3FB')
ax.set_title(
    'Tabla A1. Modelo A — Métricas de evaluación (Explicativo sin lag-1)\n'
    'Nota: R² más bajo que los modelos predictivos es esperado y correcto',
    fontsize=11, pad=10
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_01_tabla_metricas.png')
plt.close()

# ── Fig A-02: Real vs. Predicho ───────────────────────────────────────────────
print("  Fig A-02 — Real vs. Predicho")
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, y_pred_test, alpha=0.6, s=35, color=C['azul'],
           edgecolors='white', linewidths=0.4)
lm = min(y_test.min(), y_pred_test.min()) * 0.85
lM = max(y_test.max(), y_pred_test.max()) * 1.05
ax.plot([lm, lM], [lm, lM], 'k--', lw=1.5, alpha=0.5, label='Predicción perfecta')
ax.text(0.05, 0.95,
        f"R²={m_te['R2']:.4f}\nRMSE={m_te['RMSE']:.0f}\nMAE={m_te['MAE']:.0f}\nMAPE={m_te['MAPE']:.1f}%",
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=C['azul']),
        fontweight='bold', color=C['azul_osc'])
ax.set_xlabel('Valor real (tasa delitos / 100k)')
ax.set_ylabel('Predicción (tasa delitos / 100k)')
ax.set_title('Ilustración A2. Real vs. Predicho — Modelo A (test 2022-2024)', fontsize=12, pad=10)
ax.legend(fontsize=10)
ax.set_xlim(lm, lM)
ax.set_ylim(lm, lM)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_02_scatter.png')
plt.close()

# ── Fig A-03: Coeficientes β ──────────────────────────────────────────────────
print("  Fig A-03 — Coeficientes β")
fig, ax = plt.subplots(figsize=(10, 5))
coefs = pd.Series(lr.coef_, index=X_COLS)
colors_bar = [C['azul'] if v > 0 else C['rojo'] for v in coefs.values]
bars = ax.barh(
    [LABEL[v] for v in coefs.index], coefs.values,
    color=colors_bar, alpha=0.85, height=0.5
)
ax.axvline(0, color='black', lw=1.2)
for b, v in zip(bars, coefs.values):
    offset = abs(coefs.values).max() * 0.03
    ha = 'left' if v >= 0 else 'right'
    ax.text(v + np.sign(v) * offset, b.get_y() + b.get_height() / 2,
            f'{v:+.1f}', va='center', ha=ha, fontsize=11, fontweight='bold')
ax.set_xlabel('Coeficiente estandarizado (β) — indica el efecto parcial de cada variable')
ax.set_title(
    'Ilustración A3. Coeficientes β — Modelo A (Regresión Lineal, sin lag-1)\n'
    'Verde = aumenta criminalidad  |  Rojo = reduce criminalidad',
    fontsize=12, pad=10
)
# Leyenda interpretativa
ax.text(0.98, 0.04,
        "β>0: a mayor X, más delitos\nβ<0: a mayor X, menos delitos",
        transform=ax.transAxes, va='bottom', ha='right', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_03_coeficientes.png')
plt.close()

# ── Fig A-04: Evolución temporal ──────────────────────────────────────────────
print("  Fig A-04 — Evolución temporal")
X_all    = master[X_COLS].values
X_all_sc = scaler.transform(X_all)
master_p = master.copy()
master_p['Pred'] = lr.predict(X_all_sc)
evol = master_p.groupby('Año').agg({Y_COL: 'mean', 'Pred': 'mean'}).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(evol['Año'], evol[Y_COL], 'ko-', lw=2.5, ms=7, label='Real', zorder=5)
ax.plot(evol['Año'], evol['Pred'], color=C['azul'], lw=2, ls='--',
        marker='s', ms=5, label='Modelo A (Explicativo)', alpha=0.85)
ax.axvspan(YEAR_SPLIT - 0.3, evol['Año'].max() + 0.3,
           alpha=0.08, color=C['naranja'], label=f'Test ({YEAR_SPLIT}-2024)')
ax.axvline(YEAR_SPLIT - 0.5, color=C['naranja'], lw=2, ls=':', alpha=0.6)
ax.text(YEAR_SPLIT + 0.15, evol[Y_COL].max() * 0.98, 'TEST',
        fontsize=12, fontweight='bold', color=C['naranja'], va='top')
for yr in [2020, 2021]:
    ax.axvspan(yr - 0.3, yr + 0.3, alpha=0.08, color=C['rojo'], zorder=0)
ax.set_xlabel('Año')
ax.set_ylabel('Tasa media delitos / 100k hab.')
ax.set_xticks(evol['Año'])
ax.set_title('Ilustración A4. Evolución real vs. predicción — Modelo A (media nacional)',
             fontsize=12, pad=10)
ax.legend(fontsize=10)
# Nota sobre shading rojo
ax.text(2020, evol[Y_COL].min() * 1.02, 'COVID', ha='center',
        fontsize=8, color=C['rojo'], alpha=0.7)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_04_evolucion.png')
plt.close()

# ── Fig A-05: Distribución de residuos ───────────────────────────────────────
print("  Fig A-05 — Residuos")
residuos = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma
ax = axes[0]
ax.hist(residuos, bins=28, color=C['azul'], alpha=0.75, edgecolor='white')
ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.5, label='Cero ideal')
ax.axvline(residuos.mean(), color=C['rojo'], lw=2,
           label=f'Media = {residuos.mean():.1f}')
ax.text(0.97, 0.95, f"σ = {residuos.std():.1f}",
        transform=ax.transAxes, va='top', ha='right', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Residuo (real − predicho)')
ax.set_ylabel('Frecuencia')
ax.set_title('a) Histograma de residuos')
ax.legend(fontsize=9)

# QQ-like: residuos vs. índice ordenado
ax2 = axes[1]
sorted_res = np.sort(residuos)
n = len(sorted_res)
quantiles = np.linspace(0, 1, n)
ax2.scatter(quantiles, sorted_res, alpha=0.5, s=20, color=C['azul'])
ax2.axhline(0, color='black', lw=1.2, ls='--', alpha=0.6)
ax2.set_xlabel('Cuantil acumulado')
ax2.set_ylabel('Residuo ordenado')
ax2.set_title('b) Distribución acumulada de residuos')

plt.suptitle('Ilustración A5. Análisis de residuos — Modelo A (test 2022-2024)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_05_residuos.png')
plt.close()

# ── Fig A-06: Walk-Forward CV por fold ────────────────────────────────────────
print("  Fig A-06 — Walk-Forward CV")
fold_labels = [f"F{i+1}" for i in range(len(wf_splits))]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
bars_r2 = ax1.bar(fold_labels, cv_r2, color=C['azul'], alpha=0.8)
ax1.axhline(cv_r2.mean(), color=C['rojo'], lw=2, ls='--',
            label=f'Media: {cv_r2.mean():.3f}')
ax1.axhline(0, color='grey', lw=0.8)
for b, v in zip(bars_r2, cv_r2):
    ax1.text(b.get_x() + b.get_width() / 2,
             (v + 0.02 if v >= 0 else v - 0.06),
             f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')
ax1.set_ylabel('R²')
ax1.set_title('a) R² por fold walk-forward')
ax1.legend(fontsize=9)

ax2 = axes[1]
bars_rmse = ax2.bar(fold_labels, cv_rmse, color=C['naranja'], alpha=0.8)
ax2.axhline(cv_rmse.mean(), color=C['rojo'], lw=2, ls='--',
            label=f'Media: {cv_rmse.mean():.1f}')
for b, v in zip(bars_rmse, cv_rmse):
    ax2.text(b.get_x() + b.get_width() / 2, v + cv_rmse.max() * 0.02,
             f'{v:.0f}', ha='center', fontsize=8.5, fontweight='bold')
ax2.set_ylabel('RMSE')
ax2.set_title('b) RMSE por fold')
ax2.legend(fontsize=9)

plt.suptitle('Ilustración A6. Walk-Forward CV — Modelo A (Explicativo)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_06_cv_folds.png')
plt.close()

# ── Fig A-07: Dependencia Parcial ─────────────────────────────────────────────
# Para la Regresión Lineal los PDPs son exactamente las rectas de los
# coeficientes, pero es útil visualizarlos para comparar con el PDP del XGBoost
print("  Fig A-07 — PDP")
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
pdp_colors = [C['azul'], C['naranja'], C['verde'], C['rojo']]

for ax, (i, col) in zip(axes, enumerate(X_COLS)):
    if col == 'Dummy_COVID':
        means = []
        for v in [0, 1]:
            Xt = X_train_sc.copy()
            Xt[:, i] = (v - scaler.mean_[i]) / scaler.scale_[i]
            means.append(lr.predict(Xt).mean())
        ax.bar(['No COVID\n(0)', 'COVID\n(1)'], means,
               color=[C['azul'], C['rojo']], alpha=0.8, width=0.4)
        for j, val in enumerate(means):
            ax.text(j, val + abs(means[1] - means[0]) * 0.05,
                    f'{val:.0f}', ha='center', fontsize=11, fontweight='bold')
    else:
        raw_grid = np.linspace(X_train[:, i].min(), X_train[:, i].max(), 60)
        pdp = []
        for g in raw_grid:
            Xt = X_train_sc.copy()
            Xt[:, i] = (g - scaler.mean_[i]) / scaler.scale_[i]
            pdp.append(lr.predict(Xt).mean())
        ax.plot(raw_grid, pdp, color=pdp_colors[i], lw=2.5)
        ax.fill_between(raw_grid, min(pdp), pdp, alpha=0.1, color=pdp_colors[i])
        # Rug plot: distribución real de los datos
        ax.scatter(X_train[:, i], [min(pdp)] * len(X_train),
                   alpha=0.04, s=5, color='black', marker='|')
    ax.set_xlabel(LABEL[col], fontsize=9)
    ax.set_ylabel('Pred. media', fontsize=9)
    ax.set_title(col, fontsize=10)

plt.suptitle(
    'Ilustración A7. Dependencia Parcial (PDP) — Modelo A\n'
    '(relación marginal de cada variable socioeconómica con la criminalidad)',
    fontsize=12, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_A_07_pdp.png')
plt.close()

# ==============================================================================
# 6. RESUMEN FINAL
# ==============================================================================
print(f"\n{'='*72}")
print("  RESUMEN FINAL — MODELO A (EXPLICATIVO)")
print("="*72)
print(f"""
  Test:  R²={m_te['R2']:.4f}  RMSE={m_te['RMSE']:.0f}  MAE={m_te['MAE']:.0f}  MAPE={m_te['MAPE']:.1f}%
  Train: R²={m_tr['R2']:.4f}  RMSE={m_tr['RMSE']:.0f}  MAE={m_tr['MAE']:.0f}  MAPE={m_tr['MAPE']:.1f}%
  WF-CV: R²={cv_r2.mean():.4f} ± {cv_r2.std():.4f}

  Coeficientes (estandarizados):""")
for col, b in zip(X_COLS, lr.coef_):
    print(f"    {LABEL[col]:30s}  β = {b:+.2f}")
print(f"""
  INTERPRETACIÓN:
  - Este modelo responde a la pregunta EXPLICATIVA: ¿qué variables
    socioeconómicas se asocian con la criminalidad provincial?
  - El R² más bajo respecto a los modelos predictivos (B y C) es ESPERADO:
    sin el lag-1, el modelo no puede apoyarse en la inercia temporal.
  - Los coeficientes son interpretables causalmente como efectos parciales.

  Figuras generadas en {OUT}/:""")
for f in sorted(os.listdir(OUT)):
    print(f"    {f}")

  MODELO A — EXPLICATIVO (Regresión Lineal sin lag-1)

Dataset: 780 filas × 17 columnas
Provincias: 52 | Años: 2010-2024

NOTA: Modelo explicativo puro — sin lag-1, sin delta paro.
Dataset completo: 780 observaciones (2010-2024)

Partición temporal:
  Train: 624 obs  (años 2010-2021)
  Test:  156 obs  (años 2022-2024)

Walk-Forward CV: 8 folds
  Fold 1: train 2010-2013 → val 2014
  Fold 2: train 2010-2014 → val 2015
  Fold 3: train 2010-2015 → val 2016
  Fold 4: train 2010-2016 → val 2017
  Fold 5: train 2010-2017 → val 2018
  Fold 6: train 2010-2018 → val 2019
  Fold 7: train 2010-2019 → val 2020
  Fold 8: train 2010-2020 → val 2021

────────────────────────────────────────────────────────────────────────
  ENTRENAMIENTO
────────────────────────────────────────────────────────────────────────

Coeficientes estandarizados (β):
  Tasa de Paro (X₁)               β =  +213.47  (↑ más delitos)
  Renta Neta Media (X₂)           β =  +104.12  (↑ más delitos)
  Densidad Poblacional (X₃)      

In [2]:
"""
================================================================================
PILAR 2 — MODELO B: PREDICTIVO (Regresión Lineal con lag-1)
================================================================================
TFG: Criminalidad patrimonial y factores socioeconómicos en España (2010-2024)
Grado en Business Analytics — UFV 2025-26

OBJETIVO DE ESTE MODELO:
  Responder a la pregunta PREDICTIVA con un modelo de caja blanca:
  ¿Cuántos delitos habrá el año que viene en cada provincia?

  El lag-1 de la variable objetivo captura la inercia temporal de la
  criminalidad provincial. El delta de paro añade la señal de shocks
  económicos abruptos (subidas/bajadas bruscas del desempleo).

  Al ser lineal, sus coeficientes siguen siendo interpretables, lo que
  permite comparar directamente con el Modelo A para ver cuánto peso
  pierde cada variable socioeconómica al competir con el lag-1.

VARIABLES:
  Y    : Tasa_Delitos_100k
  X1   : Tasa_Paro
  X2   : Renta_Neta_Media
  X3   : Densidad_Poblacion
  X4   : Dummy_COVID
  X5   : Tasa_Delitos_100k_lag1   ← inercia temporal
  X6   : Delta_Paro               ← variación interanual del paro

FIGURAS GENERADAS (carpeta ./figuras_modeloB/):
  fig_B_01_tabla_metricas.png   — Métricas train/test + CV
  fig_B_02_scatter.png          — Real vs. Predicho en test
  fig_B_03_coeficientes.png     — Coeficientes β (con y sin lag para comparar)
  fig_B_04_evolucion.png        — Evolución temporal real vs. predicho
  fig_B_05_residuos.png         — Distribución de residuos en test
  fig_B_06_cv_folds.png         — Walk-Forward CV por fold
  fig_B_07_comparacion_coefs.png — Comparación β Modelo A vs. Modelo B
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings, os
warnings.filterwarnings('ignore')

# ── Estilo global ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})
C = {
    'azul':     '#2E75B6',
    'azul_osc': '#1F497D',
    'gris':     '#595959',
    'naranja':  '#E36C09',
    'verde':    '#375623',
    'rojo':     '#C0392B',
    'purpura':  '#8E44AD',
    'oscuro':   '#2C3E50',
    'claro':    '#BDD7EE',
}

OUT = './figuras_modeloB'
os.makedirs(OUT, exist_ok=True)

# ==============================================================================
# 1. CARGA DE DATOS Y FEATURE ENGINEERING TEMPORAL
# ==============================================================================
print("=" * 72)
print("  MODELO B — PREDICTIVO LINEAL (Regresión Lineal con lag-1)")
print("=" * 72)

master = pd.read_csv('base_datos_maestra.csv')
print(f"\nDataset original: {master.shape[0]} filas × {master.shape[1]} columnas")

Y_COL  = 'Tasa_Delitos_100k'
X_BASE = ['Tasa_Paro', 'Renta_Neta_Media', 'Densidad_Poblacion', 'Dummy_COVID']

# ── Feature Engineering Temporal ─────────────────────────────────────────────
master = master.sort_values(['Provincia', 'Año']).reset_index(drop=True)

# Lag-1: la tasa de delitos del año anterior es el mejor predictor "mecánico"
# de la tasa actual, pues la criminalidad tiene fuerte inercia estructural.
master['Tasa_Delitos_100k_lag1'] = (
    master.groupby('Provincia')[Y_COL].shift(1)
)

# Delta Paro: no solo importa el nivel del paro sino si está subiendo o bajando.
# Un incremento brusco (ej. crisis 2008-2012, COVID 2020) tiene un impacto
# distinto a un paro alto pero estable.
master['Delta_Paro'] = (
    master.groupby('Provincia')['Tasa_Paro'].diff(1)
)

filas_antes = len(master)
master = master.dropna(subset=['Tasa_Delitos_100k_lag1', 'Delta_Paro']).reset_index(drop=True)
print(f"Filas eliminadas por lag-1 (año 2010 sin histórico): {filas_antes - len(master)}")
print(f"Dataset resultante: {master.shape[0]} filas ({master['Año'].min()}-{master['Año'].max()})")

X_COLS = X_BASE + ['Tasa_Delitos_100k_lag1', 'Delta_Paro']
LABEL  = {
    'Tasa_Paro':                 'Tasa de Paro (X₁)',
    'Renta_Neta_Media':          'Renta Neta Media (X₂)',
    'Densidad_Poblacion':        'Densidad Poblacional (X₃)',
    'Dummy_COVID':               'Dummy COVID (X₄)',
    'Tasa_Delitos_100k_lag1':    'Tasa Delitos t−1 (X₅ — lag₁)',
    'Delta_Paro':                'Δ Tasa Paro (X₆)',
}

# ==============================================================================
# 2. PARTICIÓN TEMPORAL
# ==============================================================================
YEAR_SPLIT = 2022

train = master[master['Año'] <  YEAR_SPLIT].copy()
test  = master[master['Año'] >= YEAR_SPLIT].copy()

X_train = train[X_COLS].values
y_train = train[Y_COL].values
X_test  = test[X_COLS].values
y_test  = test[Y_COL].values

print(f"\nPartición temporal:")
print(f"  Train: {len(train)} obs  ({train['Año'].min()}-{train['Año'].max()})")
print(f"  Test:  {len(test)} obs  ({test['Año'].min()}-{test['Año'].max()})")

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ==============================================================================
# 3. WALK-FORWARD CV
# ==============================================================================
años_train = sorted(train['Año'].unique())
MIN_AÑOS   = 4

wf_splits = []
for val_year in años_train:
    train_years = [y for y in años_train if y < val_year]
    if len(train_years) < MIN_AÑOS:
        continue
    tr_idx = train[train['Año'].isin(train_years)].index.values
    va_idx = train[train['Año'] == val_year].index.values
    wf_splits.append((tr_idx, va_idx))

print(f"\nWalk-Forward CV: {len(wf_splits)} folds")

# ==============================================================================
# 4. ENTRENAMIENTO
# ==============================================================================
print(f"\n{'─'*72}")
print("  ENTRENAMIENTO")
print(f"{'─'*72}")

lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_train = lr.predict(X_train_sc)
y_pred_test  = lr.predict(X_test_sc)

print("\nCoeficientes estandarizados (β):")
for col, b in zip(X_COLS, lr.coef_):
    print(f"  {LABEL[col]:35s}  β = {b:+8.2f}")
print(f"  {'Intercepto':35s}  β₀= {lr.intercept_:8.2f}")

# Walk-Forward CV
def calc_metricas(y_r, y_p):
    mask = y_r != 0
    return {
        'RMSE': np.sqrt(mean_squared_error(y_r, y_p)),
        'MAE':  mean_absolute_error(y_r, y_p),
        'R2':   r2_score(y_r, y_p),
        'MAPE': np.mean(np.abs((y_r[mask] - y_p[mask]) / y_r[mask])) * 100,
    }

cv_r2, cv_rmse = [], []
for tr_idx, va_idx in wf_splits:
    tr_pos = np.where(np.isin(train.index.values, tr_idx))[0]
    va_pos = np.where(np.isin(train.index.values, va_idx))[0]
    sc_fold = StandardScaler()
    Xtr = sc_fold.fit_transform(X_train[tr_pos])
    Xva = sc_fold.transform(X_train[va_pos])
    m   = LinearRegression().fit(Xtr, y_train[tr_pos])
    yp  = m.predict(Xva)
    cv_r2.append(r2_score(y_train[va_pos], yp))
    cv_rmse.append(np.sqrt(mean_squared_error(y_train[va_pos], yp)))

cv_r2   = np.array(cv_r2)
cv_rmse = np.array(cv_rmse)
print(f"\nWalk-Forward CV ({len(wf_splits)} folds):")
print(f"  R²:   {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"  RMSE: {cv_rmse.mean():.1f} ± {cv_rmse.std():.1f}")

m_tr = calc_metricas(y_train, y_pred_train)
m_te = calc_metricas(y_test,  y_pred_test)

print(f"\nMétricas finales:")
print(f"  {'':20s} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'MAPE':>8}")
print(f"  {'Train':20s} {m_tr['RMSE']:>8.1f} {m_tr['MAE']:>8.1f} {m_tr['R2']:>8.4f} {m_tr['MAPE']:>7.1f}%")
print(f"  {'Test':20s}  {m_te['RMSE']:>8.1f} {m_te['MAE']:>8.1f} {m_te['R2']:>8.4f} {m_te['MAPE']:>7.1f}%")

# ==============================================================================
# 5. FIGURAS
# ==============================================================================
print(f"\n{'─'*72}")
print("  GENERANDO FIGURAS")
print(f"{'─'*72}")

# ── Fig B-01: Tabla métricas ──────────────────────────────────────────────────
print("  Fig B-01 — Tabla métricas")
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.axis('off')
header = ['Modelo', 'RMSE\nTrain', 'RMSE\nTest', 'MAE\nTrain', 'MAE\nTest',
          'R²\nTrain', 'R²\nTest', 'MAPE\nTrain', 'MAPE\nTest', 'WF-CV\nR² (μ±σ)']
row = [
    'Modelo B\n(Pred. Lineal)',
    f"{m_tr['RMSE']:.1f}", f"{m_te['RMSE']:.1f}",
    f"{m_tr['MAE']:.1f}",  f"{m_te['MAE']:.1f}",
    f"{m_tr['R2']:.4f}",   f"{m_te['R2']:.4f}",
    f"{m_tr['MAPE']:.1f}%", f"{m_te['MAPE']:.1f}%",
    f"{cv_r2.mean():.3f}±{cv_r2.std():.3f}",
]
t = ax.table(cellText=[row], colLabels=header, cellLoc='center',
             loc='center', bbox=[0, 0.1, 1, 0.85])
t.auto_set_font_size(False)
t.set_fontsize(10)
for j in range(len(header)):
    t[0, j].set_facecolor(C['azul'])
    t[0, j].set_text_props(color='white', fontweight='bold', fontsize=9)
    t[1, j].set_facecolor('#EBF3FB')
ax.set_title('Tabla B1. Modelo B — Métricas de evaluación (Predictivo Lineal con lag-1)',
             fontsize=11, pad=10)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_01_tabla_metricas.png')
plt.close()

# ── Fig B-02: Real vs. Predicho ───────────────────────────────────────────────
print("  Fig B-02 — Real vs. Predicho")
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, y_pred_test, alpha=0.6, s=35, color=C['azul'],
           edgecolors='white', linewidths=0.4)
lm = min(y_test.min(), y_pred_test.min()) * 0.85
lM = max(y_test.max(), y_pred_test.max()) * 1.05
ax.plot([lm, lM], [lm, lM], 'k--', lw=1.5, alpha=0.5, label='Predicción perfecta')
ax.text(0.05, 0.95,
        f"R²={m_te['R2']:.4f}\nRMSE={m_te['RMSE']:.0f}\nMAE={m_te['MAE']:.0f}\nMAPE={m_te['MAPE']:.1f}%",
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=C['azul']),
        fontweight='bold', color=C['azul_osc'])
ax.set_xlabel('Valor real (tasa delitos / 100k)')
ax.set_ylabel('Predicción (tasa delitos / 100k)')
ax.set_title('Ilustración B2. Real vs. Predicho — Modelo B (test 2022-2024)', fontsize=12, pad=10)
ax.legend(fontsize=10)
ax.set_xlim(lm, lM)
ax.set_ylim(lm, lM)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_02_scatter.png')
plt.close()

# ── Fig B-03: Coeficientes β ──────────────────────────────────────────────────
print("  Fig B-03 — Coeficientes β")
fig, ax = plt.subplots(figsize=(11, 6))
coefs = pd.Series(lr.coef_, index=X_COLS)
color_map = {
    'Tasa_Paro':              C['azul'],
    'Renta_Neta_Media':       C['naranja'],
    'Densidad_Poblacion':     C['verde'],
    'Dummy_COVID':            C['rojo'],
    'Tasa_Delitos_100k_lag1': C['purpura'],
    'Delta_Paro':             C['oscuro'],
}
colors_bar = [color_map[v] for v in coefs.index]
bars = ax.barh([LABEL[v] for v in coefs.index], coefs.values,
               color=colors_bar, alpha=0.85, height=0.5)
ax.axvline(0, color='black', lw=1.2)
for b, v in zip(bars, coefs.values):
    offset = abs(coefs.values).max() * 0.02
    ha = 'left' if v >= 0 else 'right'
    ax.text(v + np.sign(v) * offset, b.get_y() + b.get_height() / 2,
            f'{v:+.1f}', va='center', ha=ha, fontsize=10, fontweight='bold')
ax.set_xlabel('Coeficiente estandarizado (β)')
ax.set_title(
    'Ilustración B3. Coeficientes β — Modelo B (Predictivo Lineal con lag-1)\n'
    'Nota: el lag-1 domina — las variables socioeconómicas tienen efecto marginal',
    fontsize=12, pad=10
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_03_coeficientes.png')
plt.close()

# ── Fig B-04: Evolución temporal ──────────────────────────────────────────────
print("  Fig B-04 — Evolución temporal")
X_all    = master[X_COLS].values
X_all_sc = scaler.transform(X_all)
master_p = master.copy()
master_p['Pred'] = lr.predict(X_all_sc)
evol = master_p.groupby('Año').agg({Y_COL: 'mean', 'Pred': 'mean'}).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(evol['Año'], evol[Y_COL], 'ko-', lw=2.5, ms=7, label='Real', zorder=5)
ax.plot(evol['Año'], evol['Pred'], color=C['azul'], lw=2, ls='--',
        marker='s', ms=5, label='Modelo B (Pred. Lineal)', alpha=0.85)
ax.axvspan(YEAR_SPLIT - 0.3, evol['Año'].max() + 0.3,
           alpha=0.08, color=C['naranja'], label=f'Test ({YEAR_SPLIT}-2024)')
ax.axvline(YEAR_SPLIT - 0.5, color=C['naranja'], lw=2, ls=':', alpha=0.6)
ax.text(YEAR_SPLIT + 0.15, evol[Y_COL].max() * 0.98, 'TEST',
        fontsize=12, fontweight='bold', color=C['naranja'], va='top')
for yr in [2020, 2021]:
    ax.axvspan(yr - 0.3, yr + 0.3, alpha=0.08, color=C['rojo'], zorder=0)
ax.set_xlabel('Año')
ax.set_ylabel('Tasa media delitos / 100k hab.')
ax.set_xticks(evol['Año'])
ax.set_title('Ilustración B4. Evolución real vs. predicción — Modelo B (media nacional)',
             fontsize=12, pad=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_04_evolucion.png')
plt.close()

# ── Fig B-05: Residuos ────────────────────────────────────────────────────────
print("  Fig B-05 — Residuos")
residuos = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(residuos, bins=28, color=C['azul'], alpha=0.75, edgecolor='white')
ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.5, label='Cero ideal')
ax.axvline(residuos.mean(), color=C['rojo'], lw=2,
           label=f'Media = {residuos.mean():.1f}')
ax.text(0.97, 0.95, f"σ = {residuos.std():.1f}",
        transform=ax.transAxes, va='top', ha='right', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Residuo (real − predicho)')
ax.set_ylabel('Frecuencia')
ax.set_title('a) Histograma de residuos')
ax.legend(fontsize=9)

ax2 = axes[1]
sorted_res = np.sort(residuos)
ax2.scatter(np.linspace(0, 1, len(sorted_res)), sorted_res,
            alpha=0.5, s=20, color=C['azul'])
ax2.axhline(0, color='black', lw=1.2, ls='--', alpha=0.6)
ax2.set_xlabel('Cuantil acumulado')
ax2.set_ylabel('Residuo ordenado')
ax2.set_title('b) Distribución acumulada de residuos')

plt.suptitle('Ilustración B5. Análisis de residuos — Modelo B (test 2022-2024)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_05_residuos.png')
plt.close()

# ── Fig B-06: Walk-Forward CV por fold ────────────────────────────────────────
print("  Fig B-06 — Walk-Forward CV")
fold_labels = [f"F{i+1}" for i in range(len(wf_splits))]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
bars_r2 = ax1.bar(fold_labels, cv_r2, color=C['azul'], alpha=0.8)
ax1.axhline(cv_r2.mean(), color=C['rojo'], lw=2, ls='--',
            label=f'Media: {cv_r2.mean():.3f}')
ax1.axhline(0, color='grey', lw=0.8)
for b, v in zip(bars_r2, cv_r2):
    ax1.text(b.get_x() + b.get_width() / 2,
             (v + 0.02 if v >= 0 else v - 0.06),
             f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')
ax1.set_ylabel('R²')
ax1.set_title('a) R² por fold walk-forward')
ax1.legend(fontsize=9)

ax2 = axes[1]
bars_rmse = ax2.bar(fold_labels, cv_rmse, color=C['naranja'], alpha=0.8)
ax2.axhline(cv_rmse.mean(), color=C['rojo'], lw=2, ls='--',
            label=f'Media: {cv_rmse.mean():.1f}')
for b, v in zip(bars_rmse, cv_rmse):
    ax2.text(b.get_x() + b.get_width() / 2, v + cv_rmse.max() * 0.02,
             f'{v:.0f}', ha='center', fontsize=8.5, fontweight='bold')
ax2.set_ylabel('RMSE')
ax2.set_title('b) RMSE por fold')
ax2.legend(fontsize=9)

plt.suptitle('Ilustración B6. Walk-Forward CV — Modelo B (Predictivo Lineal)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_06_cv_folds.png')
plt.close()

# ── Fig B-07: Comparación coefs Modelo A vs. Modelo B ────────────────────────
# Esta figura es clave para el argumento central del TFG:
# las variables económicas pierden peso al añadir el lag-1
print("  Fig B-07 — Comparación coeficientes A vs. B")

# Reentrenar Modelo A sobre los mismos años (2011-2021, sin 2010 por lag)
# para que la comparación sea justa (mismo dataset)
X_COLS_A = ['Tasa_Paro', 'Renta_Neta_Media', 'Densidad_Poblacion', 'Dummy_COVID']
scaler_A = StandardScaler()
Xtr_A = scaler_A.fit_transform(train[X_COLS_A].values)
lr_A  = LinearRegression().fit(Xtr_A, y_train)
coefs_A = pd.Series(lr_A.coef_, index=X_COLS_A)

# Variables comunes (las 4 socioeconómicas)
vars_comunes = X_COLS_A
labels_comunes = [LABEL[v] for v in vars_comunes]

coefs_B_comunes = pd.Series(lr.coef_[:4], index=X_COLS_A)

x_pos = np.arange(len(vars_comunes))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars_A = ax.bar(x_pos - width/2, coefs_A.values, width,
                color=C['naranja'], alpha=0.85, label='Modelo A (sin lag-1)')
bars_B = ax.bar(x_pos + width/2, coefs_B_comunes.values, width,
                color=C['azul'], alpha=0.85, label='Modelo B (con lag-1)')

ax.axhline(0, color='black', lw=1)
for bars in [bars_A, bars_B]:
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2,
                v + (2 if v >= 0 else -5),
                f'{v:+.1f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(labels_comunes, fontsize=10)
ax.set_ylabel('Coeficiente estandarizado (β)')
ax.legend(fontsize=10)
ax.set_title(
    'Ilustración B7. Impacto del lag-1 sobre los coeficientes de las variables socioeconómicas\n'
    'Al añadir el lag-1 (Modelo B), las variables económicas reducen su peso — el lag se "lleva" su señal',
    fontsize=11, pad=10
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_B_07_comparacion_coefs.png')
plt.close()

# ==============================================================================
# 6. RESUMEN FINAL
# ==============================================================================
print(f"\n{'='*72}")
print("  RESUMEN FINAL — MODELO B (PREDICTIVO LINEAL)")
print("="*72)
print(f"""
  Test:  R²={m_te['R2']:.4f}  RMSE={m_te['RMSE']:.0f}  MAE={m_te['MAE']:.0f}  MAPE={m_te['MAPE']:.1f}%
  Train: R²={m_tr['R2']:.4f}  RMSE={m_tr['RMSE']:.0f}  MAE={m_tr['MAE']:.0f}  MAPE={m_tr['MAPE']:.1f}%
  WF-CV: R²={cv_r2.mean():.4f} ± {cv_r2.std():.4f}

  Coeficientes (estandarizados):""")
for col, b in zip(X_COLS, lr.coef_):
    print(f"    {LABEL[col]:35s}  β = {b:+.2f}")

print(f"""
  INTERPRETACIÓN:
  - El lag-1 domina con diferencia — confirma la inercia temporal
    de la criminalidad provincial en España.
  - La figura B7 muestra cómo los coeficientes de las variables
    económicas se reducen al añadir el lag-1 (el lag "absorbe" su señal).
  - Para el efecto causal de las variables económicas, véase Modelo A.

  Figuras generadas en {OUT}/:""")
for f in sorted(os.listdir(OUT)):
    print(f"    {f}")

  MODELO B — PREDICTIVO LINEAL (Regresión Lineal con lag-1)

Dataset original: 780 filas × 17 columnas
Filas eliminadas por lag-1 (año 2010 sin histórico): 52
Dataset resultante: 728 filas (2011-2024)

Partición temporal:
  Train: 572 obs  (2011-2021)
  Test:  156 obs  (2022-2024)

Walk-Forward CV: 7 folds

────────────────────────────────────────────────────────────────────────
  ENTRENAMIENTO
────────────────────────────────────────────────────────────────────────

Coeficientes estandarizados (β):
  Tasa de Paro (X₁)                    β =    +7.45
  Renta Neta Media (X₂)                β =   +18.53
  Densidad Poblacional (X₃)            β =    -0.13
  Dummy COVID (X₄)                     β =   -84.76
  Tasa Delitos t−1 (X₅ — lag₁)         β =  +749.83
  Δ Tasa Paro (X₆)                     β =   +15.91
  Intercepto                           β₀=  1770.49

Walk-Forward CV (7 folds):
  R²:   0.4677 ± 0.8005
  RMSE: 268.8 ± 242.2

Métricas finales:
                           RMSE      M

In [3]:
"""
================================================================================
PILAR 2 — MODELO C: PREDICTIVO AVANZADO (XGBoost con lag-1)
================================================================================
TFG: Criminalidad patrimonial y factores socioeconómicos en España (2010-2024)
Grado en Business Analytics — UFV 2025-26

OBJETIVO DE ESTE MODELO:
  Modelo predictivo de caja negra — maximizar la capacidad de forecasting
  capturando relaciones no lineales e interacciones entre variables.

  Compite directamente con el Modelo B (mismo dataset, mismas variables)
  para responder: ¿añade valor la complejidad del gradient boosting
  frente a la regresión lineal cuando la variable objetivo tiene
  una autocorrelación temporal tan alta?

VARIABLES (idénticas al Modelo B):
  Y    : Tasa_Delitos_100k
  X1   : Tasa_Paro
  X2   : Renta_Neta_Media
  X3   : Densidad_Poblacion
  X4   : Dummy_COVID
  X5   : Tasa_Delitos_100k_lag1
  X6   : Delta_Paro

MEJORAS TÉCNICAS:
  1. Validación Walk-Forward (respeta estructura temporal del panel)
  2. Optimización Bayesiana con Optuna TPE (50 trials)
     → Fallback automático a RandomSearch si Optuna no está disponible
  3. Interpretabilidad: SHAP nativo si disponible
     → Fallback a Permutation Importance + PDP 2D

FIGURAS GENERADAS (carpeta ./figuras_modeloC/):
  fig_C_01_tabla_metricas.png   — Métricas train/test + WF-CV
  fig_C_02_scatter.png          — Real vs. Predicho en test
  fig_C_03_feature_importance.png — MDI + Permutation Importance
  fig_C_04_evolucion.png        — Evolución temporal real vs. predicho
  fig_C_05_residuos.png         — Distribución de residuos en test
  fig_C_06_cv_folds.png         — Walk-Forward CV por fold
  fig_C_07_pdp.png              — PDP univariados (6 variables)
  fig_C_08_pdp2d.png            — PDP 2D: Paro × Renta (o SHAP dependence)
  fig_C_09_optim.png            — Historial de optimización de hiperparámetros
  fig_C_10_comparacion_BC.png   — Comparación Modelo B vs. Modelo C en test
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings, os, time
warnings.filterwarnings('ignore')

# ── Librerías opcionales ──────────────────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    USE_XGBOOST = True
    print("[✓] XGBoost nativo detectado")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor
    USE_XGBOOST = False
    print("[!] XGBoost no disponible → GradientBoostingRegressor (equivalente)")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    USE_OPTUNA = True
    print("[✓] Optuna detectado")
except ImportError:
    USE_OPTUNA = False
    print("[!] Optuna no disponible → RandomSearch como fallback")

try:
    import shap
    USE_SHAP = True
    print("[✓] SHAP detectado")
except ImportError:
    USE_SHAP = False
    print("[!] SHAP no disponible → Permutation Importance + PDP 2D")

# ── Estilo global ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})
C = {
    'azul':     '#2E75B6',
    'azul_osc': '#1F497D',
    'gris':     '#595959',
    'naranja':  '#E36C09',
    'verde':    '#375623',
    'rojo':     '#C0392B',
    'purpura':  '#8E44AD',
    'oscuro':   '#2C3E50',
    'claro':    '#BDD7EE',
}

OUT = './figuras_modeloC'
os.makedirs(OUT, exist_ok=True)

# ==============================================================================
# 1. CARGA DE DATOS Y FEATURE ENGINEERING
# ==============================================================================
print("\n" + "=" * 72)
print("  MODELO C — PREDICTIVO AVANZADO (XGBoost con lag-1)")
print("=" * 72)

master = pd.read_csv('base_datos_maestra.csv')
print(f"\nDataset original: {master.shape[0]} filas × {master.shape[1]} columnas")

Y_COL  = 'Tasa_Delitos_100k'
X_BASE = ['Tasa_Paro', 'Renta_Neta_Media', 'Densidad_Poblacion', 'Dummy_COVID']
LABEL  = {
    'Tasa_Paro':                 'Tasa de Paro (X₁)',
    'Renta_Neta_Media':          'Renta Neta Media (X₂)',
    'Densidad_Poblacion':        'Densidad Poblacional (X₃)',
    'Dummy_COVID':               'Dummy COVID (X₄)',
    'Tasa_Delitos_100k_lag1':    'Tasa Delitos t−1 (X₅ — lag₁)',
    'Delta_Paro':                'Δ Tasa Paro (X₆)',
}

master = master.sort_values(['Provincia', 'Año']).reset_index(drop=True)
master['Tasa_Delitos_100k_lag1'] = master.groupby('Provincia')[Y_COL].shift(1)
master['Delta_Paro'] = master.groupby('Provincia')['Tasa_Paro'].diff(1)

filas_antes = len(master)
master = master.dropna(subset=['Tasa_Delitos_100k_lag1', 'Delta_Paro']).reset_index(drop=True)
print(f"Filas eliminadas (lag-1): {filas_antes - len(master)}")
print(f"Dataset resultante: {master.shape[0]} filas ({master['Año'].min()}-{master['Año'].max()})")

X_COLS = X_BASE + ['Tasa_Delitos_100k_lag1', 'Delta_Paro']

# ==============================================================================
# 2. PARTICIÓN TEMPORAL
# ==============================================================================
YEAR_SPLIT = 2022

train = master[master['Año'] <  YEAR_SPLIT].copy()
test  = master[master['Año'] >= YEAR_SPLIT].copy()

X_train = train[X_COLS].values
y_train = train[Y_COL].values
X_test  = test[X_COLS].values
y_test  = test[Y_COL].values

print(f"\nPartición temporal:")
print(f"  Train: {len(train)} obs  ({train['Año'].min()}-{train['Año'].max()})")
print(f"  Test:  {len(test)} obs  ({test['Año'].min()}-{test['Año'].max()})")

# XGBoost no necesita escalado, pero preparamos el scaler para Modelo B
# (usado solo en la figura comparativa final)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ==============================================================================
# 3. WALK-FORWARD CV
# ==============================================================================
años_train = sorted(train['Año'].unique())
MIN_AÑOS   = 4

wf_splits = []
for val_year in años_train:
    train_years = [y for y in años_train if y < val_year]
    if len(train_years) < MIN_AÑOS:
        continue
    tr_idx = train[train['Año'].isin(train_years)].index.values
    va_idx = train[train['Año'] == val_year].index.values
    wf_splits.append((tr_idx, va_idx))

print(f"\nWalk-Forward CV: {len(wf_splits)} folds generados")
for i, (tr_idx, va_idx) in enumerate(wf_splits):
    tr_y = sorted(train.loc[tr_idx, 'Año'].unique())
    va_y = train.loc[va_idx, 'Año'].unique()[0]
    print(f"  Fold {i+1}: train {tr_y[0]}-{tr_y[-1]} → val {va_y}")

def calc_metricas(y_r, y_p):
    mask = y_r != 0
    return {
        'RMSE': np.sqrt(mean_squared_error(y_r, y_p)),
        'MAE':  mean_absolute_error(y_r, y_p),
        'R2':   r2_score(y_r, y_p),
        'MAPE': np.mean(np.abs((y_r[mask] - y_p[mask]) / y_r[mask])) * 100,
    }

def wf_eval_xgb(params, X, y, splits):
    """Evalúa una configuración XGBoost con walk-forward CV."""
    r2s = []
    for tr_idx, va_idx in splits:
        tr_pos = np.where(np.isin(train.index.values, tr_idx))[0]
        va_pos = np.where(np.isin(train.index.values, va_idx))[0]
        model = XGBRegressor(random_state=42, **params)
        model.fit(X[tr_pos], y[tr_pos])
        r2s.append(r2_score(y[va_pos], model.predict(X[va_pos])))
    return np.mean(r2s)

# ==============================================================================
# 4. OPTIMIZACIÓN DE HIPERPARÁMETROS
# ==============================================================================
print(f"\n{'─'*72}")
print("  OPTIMIZACIÓN DE HIPERPARÁMETROS")
print(f"{'─'*72}")

N_TRIALS = 50

if USE_OPTUNA:
    print(f"  Método: Optuna TPE Bayesiana ({N_TRIALS} trials)")

    def objective(trial):
        params = {
            'n_estimators':  trial.suggest_int('n_estimators', 100, 800),
            'max_depth':     trial.suggest_int('max_depth', 3, 6),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
            'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        }
        if USE_XGBOOST:
            params['min_child_weight'] = trial.suggest_int('min_child_weight', 1, 10)
            params['colsample_bytree'] = trial.suggest_float('colsample_bytree', 0.5, 1.0)
            params['reg_alpha']        = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True)
            params['reg_lambda']       = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
        else:
            params['min_samples_leaf'] = trial.suggest_int('min_samples_leaf', 3, 15)
        return wf_eval_xgb(params, X_train, y_train, wf_splits)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    elapsed = time.time() - t0

    best_params  = study.best_params
    best_cv_r2   = study.best_value
    trials_history = [
        {'trial': t.number + 1, 'cv_r2': t.value, **t.params}
        for t in study.trials if t.value is not None
    ]
    print(f"  Completado: {len(trials_history)} trials en {elapsed:.1f}s")
    print(f"  Mejor CV-R²: {best_cv_r2:.4f}")

else:
    print(f"  Método: RandomSearch ({N_TRIALS} iteraciones) + Walk-Forward manual")
    from scipy.stats import randint, uniform
    np.random.seed(42)

    param_dists = {
        'n_estimators':     randint(100, 801),
        'max_depth':        randint(3, 7),
        'learning_rate':    uniform(0.005, 0.145),
        'min_samples_leaf': randint(3, 16),
        'subsample':        uniform(0.6, 0.4),
    }

    trials_history = []
    best_cv_r2     = -np.inf
    best_params    = None
    t0 = time.time()

    for trial_num in range(N_TRIALS):
        params = {k: d.rvs() for k, d in param_dists.items()}
        params['n_estimators']     = int(params['n_estimators'])
        params['max_depth']        = int(params['max_depth'])
        params['min_samples_leaf'] = int(params['min_samples_leaf'])
        cv_r2_trial = wf_eval_xgb(params, X_train, y_train, wf_splits)
        trials_history.append({'trial': trial_num + 1, 'cv_r2': cv_r2_trial, **params})
        if cv_r2_trial > best_cv_r2:
            best_cv_r2  = cv_r2_trial
            best_params = params.copy()

    elapsed = time.time() - t0
    print(f"  Completado: {N_TRIALS} trials en {elapsed:.1f}s")
    print(f"  Mejor CV-R²: {best_cv_r2:.4f}")

print(f"\n  Mejores hiperparámetros:")
for k, v in best_params.items():
    print(f"    {k}: {v}")

# ==============================================================================
# 5. ENTRENAMIENTO DEL MODELO FINAL
# ==============================================================================
print(f"\n{'─'*72}")
print("  ENTRENAMIENTO MODELO FINAL")
print(f"{'─'*72}")

xgb = XGBRegressor(random_state=42, **best_params)
xgb.fit(X_train, y_train)

y_pred_train = xgb.predict(X_train)
y_pred_test  = xgb.predict(X_test)

m_tr = calc_metricas(y_train, y_pred_train)
m_te = calc_metricas(y_test,  y_pred_test)
overfit = m_tr['R2'] - m_te['R2']

# Walk-Forward del modelo final (con hiperparámetros óptimos)
cv_r2_list, cv_rmse_list = [], []
for tr_idx, va_idx in wf_splits:
    tr_pos = np.where(np.isin(train.index.values, tr_idx))[0]
    va_pos = np.where(np.isin(train.index.values, va_idx))[0]
    m_fold = XGBRegressor(random_state=42, **best_params)
    m_fold.fit(X_train[tr_pos], y_train[tr_pos])
    yp = m_fold.predict(X_train[va_pos])
    cv_r2_list.append(r2_score(y_train[va_pos], yp))
    cv_rmse_list.append(np.sqrt(mean_squared_error(y_train[va_pos], yp)))

cv_r2   = np.array(cv_r2_list)
cv_rmse = np.array(cv_rmse_list)

print(f"\nMétricas finales:")
print(f"  {'':20s} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'MAPE':>8}")
print(f"  {'Train':20s} {m_tr['RMSE']:>8.1f} {m_tr['MAE']:>8.1f} {m_tr['R2']:>8.4f} {m_tr['MAPE']:>7.1f}%")
print(f"  {'Test':20s}  {m_te['RMSE']:>8.1f} {m_te['MAE']:>8.1f} {m_te['R2']:>8.4f} {m_te['MAPE']:>7.1f}%")
print(f"  Sobreajuste (R² train - test): {overfit:.4f} {'⚠ revisar' if overfit > 0.15 else '✓ aceptable'}")
print(f"  WF-CV: R²={cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

# Feature Importance (MDI)
fi = pd.DataFrame({
    'Variable':    X_COLS,
    'Importancia': xgb.feature_importances_
}).sort_values('Importancia', ascending=False)

print(f"\nFeature Importance (MDI):")
for _, r in fi.iterrows():
    bar = '█' * int(r['Importancia'] * 40)
    print(f"  {LABEL[r['Variable']]:35s}: {r['Importancia']*100:.1f}%  {bar}")

# ==============================================================================
# 6. INTERPRETABILIDAD (SHAP / Permutation Importance)
# ==============================================================================
print(f"\n{'─'*72}")
print("  INTERPRETABILIDAD")
print(f"{'─'*72}")

if USE_SHAP:
    print("  Calculando valores SHAP (TreeExplainer)...")
    explainer   = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)
    perm_importances = None
else:
    from sklearn.inspection import permutation_importance
    print("  Calculando Permutation Importance (proxy de SHAP, 30 repeticiones)...")
    perm_result = permutation_importance(
        xgb, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1
    )
    perm_fi = pd.DataFrame({
        'Variable':    X_COLS,
        'Importancia': perm_result.importances_mean,
        'Std':         perm_result.importances_std,
    }).sort_values('Importancia', ascending=True)
    shap_values = None

# ==============================================================================
# 7. FIGURAS
# ==============================================================================
print(f"\n{'─'*72}")
print("  GENERANDO FIGURAS")
print(f"{'─'*72}")

# ── Fig C-01: Tabla métricas ──────────────────────────────────────────────────
print("  Fig C-01 — Tabla métricas")
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.axis('off')
header = ['Modelo', 'RMSE\nTrain', 'RMSE\nTest', 'MAE\nTrain', 'MAE\nTest',
          'R²\nTrain', 'R²\nTest', 'MAPE\nTrain', 'MAPE\nTest', 'WF-CV\nR² (μ±σ)']
row = [
    'Modelo C\n(XGBoost)',
    f"{m_tr['RMSE']:.1f}", f"{m_te['RMSE']:.1f}",
    f"{m_tr['MAE']:.1f}",  f"{m_te['MAE']:.1f}",
    f"{m_tr['R2']:.4f}",   f"{m_te['R2']:.4f}",
    f"{m_tr['MAPE']:.1f}%", f"{m_te['MAPE']:.1f}%",
    f"{cv_r2.mean():.3f}±{cv_r2.std():.3f}",
]
t = ax.table(cellText=[row], colLabels=header, cellLoc='center',
             loc='center', bbox=[0, 0.1, 1, 0.85])
t.auto_set_font_size(False)
t.set_fontsize(10)
for j in range(len(header)):
    t[0, j].set_facecolor(C['verde'])
    t[0, j].set_text_props(color='white', fontweight='bold', fontsize=9)
    t[1, j].set_facecolor('#E8F5E9')
ax.set_title(
    f'Tabla C1. Modelo C — Métricas de evaluación (XGBoost con lag-1)\n'
    f'Optuna: {"Sí" if USE_OPTUNA else "No (RandomSearch)"} | '
    f'SHAP: {"Sí" if USE_SHAP else "No (Permutation Importance)"}',
    fontsize=11, pad=10
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_01_tabla_metricas.png')
plt.close()

# ── Fig C-02: Real vs. Predicho ───────────────────────────────────────────────
print("  Fig C-02 — Real vs. Predicho")
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, y_pred_test, alpha=0.6, s=35, color=C['verde'],
           edgecolors='white', linewidths=0.4)
lm = min(y_test.min(), y_pred_test.min()) * 0.85
lM = max(y_test.max(), y_pred_test.max()) * 1.05
ax.plot([lm, lM], [lm, lM], 'k--', lw=1.5, alpha=0.5, label='Predicción perfecta')
ax.text(0.05, 0.95,
        f"R²={m_te['R2']:.4f}\nRMSE={m_te['RMSE']:.0f}\nMAE={m_te['MAE']:.0f}\nMAPE={m_te['MAPE']:.1f}%",
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=C['verde']),
        fontweight='bold', color=C['azul_osc'])
ax.set_xlabel('Valor real (tasa delitos / 100k)')
ax.set_ylabel('Predicción (tasa delitos / 100k)')
ax.set_title('Ilustración C2. Real vs. Predicho — Modelo C (test 2022-2024)', fontsize=12, pad=10)
ax.legend(fontsize=10)
ax.set_xlim(lm, lM)
ax.set_ylim(lm, lM)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_02_scatter.png')
plt.close()

# ── Fig C-03: Feature Importance MDI + Permutation / SHAP ────────────────────
print("  Fig C-03 — Feature Importance")
fi_sorted = fi.sort_values('Importancia', ascending=True)
color_map = {
    'Tasa_Paro':              C['azul'],
    'Renta_Neta_Media':       C['naranja'],
    'Densidad_Poblacion':     C['verde'],
    'Dummy_COVID':            C['rojo'],
    'Tasa_Delitos_100k_lag1': C['purpura'],
    'Delta_Paro':             C['oscuro'],
}

if USE_SHAP:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # MDI
    ax = axes[0]
    bars = ax.barh(
        [LABEL.get(v, v) for v in fi_sorted['Variable']],
        fi_sorted['Importancia'],
        color=[color_map.get(v, C['gris']) for v in fi_sorted['Variable']],
        alpha=0.85, height=0.5
    )
    for b, v in zip(bars, fi_sorted['Importancia']):
        ax.text(b.get_width() + 0.005, b.get_y() + b.get_height() / 2,
                f'{v*100:.1f}%', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Importancia MDI')
    ax.set_title('a) Feature Importance (MDI)', fontsize=11)
    # SHAP mean abs
    ax2 = axes[1]
    shap_mean = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame({'Variable': X_COLS, 'SHAP': shap_mean}).sort_values('SHAP', ascending=True)
    bars2 = ax2.barh(
        [LABEL.get(v, v) for v in shap_df['Variable']],
        shap_df['SHAP'],
        color=[color_map.get(v, C['gris']) for v in shap_df['Variable']],
        alpha=0.85, height=0.5
    )
    for b, v in zip(bars2, shap_df['SHAP']):
        ax2.text(b.get_width() + shap_df['SHAP'].max() * 0.02,
                 b.get_y() + b.get_height() / 2,
                 f'{v:.1f}', va='center', fontsize=9, fontweight='bold')
    ax2.set_xlabel('|SHAP value| medio')
    ax2.set_title('b) SHAP — importancia media absoluta', fontsize=11)
    plt.suptitle('Ilustración C3. Importancia de variables — Modelo C (XGBoost)',
                 fontsize=13, fontweight='bold', y=1.02)
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # MDI
    ax = axes[0]
    bars = ax.barh(
        [LABEL.get(v, v) for v in fi_sorted['Variable']],
        fi_sorted['Importancia'],
        color=[color_map.get(v, C['gris']) for v in fi_sorted['Variable']],
        alpha=0.85, height=0.5
    )
    for b, v in zip(bars, fi_sorted['Importancia']):
        ax.text(b.get_width() + 0.005, b.get_y() + b.get_height() / 2,
                f'{v*100:.1f}%', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Importancia MDI')
    ax.set_title('a) Feature Importance (MDI)\nNota: biased hacia lag-1 por alta correlación', fontsize=10)
    # Permutation Importance
    ax2 = axes[1]
    perm_sorted = perm_fi
    bars2 = ax2.barh(
        [LABEL.get(v, v) for v in perm_sorted['Variable']],
        perm_sorted['Importancia'],
        xerr=perm_sorted['Std'],
        color=[color_map.get(v, C['gris']) for v in perm_sorted['Variable']],
        alpha=0.85, height=0.5, capsize=3, ecolor=C['gris']
    )
    for b, v in zip(bars2, perm_sorted['Importancia']):
        ax2.text(b.get_width() + perm_sorted['Std'].max() * 0.3,
                 b.get_y() + b.get_height() / 2,
                 f'{v:.2f}', va='center', fontsize=9, fontweight='bold')
    ax2.axvline(0, color='black', lw=0.8)
    ax2.set_xlabel('Reducción R² al permutar (±1σ, 30 repeticiones)')
    ax2.set_title('b) Permutation Importance (sobre test)\nMétrica robusta: mide impacto real', fontsize=10)
    plt.suptitle(
        'Ilustración C3. Importancia de variables — Modelo C (XGBoost)\n'
        'MDI (izq.) vs. Permutation Importance (dcha.): el lag-1 domina en ambas métricas',
        fontsize=12, fontweight='bold', y=1.02
    )

plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_03_feature_importance.png')
plt.close()

# ── Fig C-04: Evolución temporal ──────────────────────────────────────────────
print("  Fig C-04 — Evolución temporal")
X_all    = master[X_COLS].values
master_p = master.copy()
master_p['Pred'] = xgb.predict(X_all)
evol = master_p.groupby('Año').agg({Y_COL: 'mean', 'Pred': 'mean'}).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(evol['Año'], evol[Y_COL], 'ko-', lw=2.5, ms=7, label='Real', zorder=5)
ax.plot(evol['Año'], evol['Pred'], color=C['verde'], lw=2, ls='-.',
        marker='^', ms=5, label='Modelo C (XGBoost)', alpha=0.85)
ax.axvspan(YEAR_SPLIT - 0.3, evol['Año'].max() + 0.3,
           alpha=0.08, color=C['naranja'], label=f'Test ({YEAR_SPLIT}-2024)')
ax.axvline(YEAR_SPLIT - 0.5, color=C['naranja'], lw=2, ls=':', alpha=0.6)
ax.text(YEAR_SPLIT + 0.15, evol[Y_COL].max() * 0.98, 'TEST',
        fontsize=12, fontweight='bold', color=C['naranja'], va='top')
for yr in [2020, 2021]:
    ax.axvspan(yr - 0.3, yr + 0.3, alpha=0.08, color=C['rojo'], zorder=0)
ax.set_xlabel('Año')
ax.set_ylabel('Tasa media delitos / 100k hab.')
ax.set_xticks(evol['Año'])
ax.set_title('Ilustración C4. Evolución real vs. predicción — Modelo C (media nacional)',
             fontsize=12, pad=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_04_evolucion.png')
plt.close()

# ── Fig C-05: Residuos ────────────────────────────────────────────────────────
print("  Fig C-05 — Residuos")
residuos = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(residuos, bins=28, color=C['verde'], alpha=0.75, edgecolor='white')
ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.5, label='Cero ideal')
ax.axvline(residuos.mean(), color=C['rojo'], lw=2,
           label=f'Media = {residuos.mean():.1f}')
ax.text(0.97, 0.95, f"σ = {residuos.std():.1f}",
        transform=ax.transAxes, va='top', ha='right', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Residuo (real − predicho)')
ax.set_ylabel('Frecuencia')
ax.set_title('a) Histograma de residuos')
ax.legend(fontsize=9)

ax2 = axes[1]
sorted_res = np.sort(residuos)
ax2.scatter(np.linspace(0, 1, len(sorted_res)), sorted_res,
            alpha=0.5, s=20, color=C['verde'])
ax2.axhline(0, color='black', lw=1.2, ls='--', alpha=0.6)
ax2.set_xlabel('Cuantil acumulado')
ax2.set_ylabel('Residuo ordenado')
ax2.set_title('b) Distribución acumulada de residuos')

plt.suptitle('Ilustración C5. Análisis de residuos — Modelo C (test 2022-2024)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_05_residuos.png')
plt.close()

# ── Fig C-06: Walk-Forward CV por fold ────────────────────────────────────────
print("  Fig C-06 — Walk-Forward CV")
fold_labels = [f"F{i+1}" for i in range(len(wf_splits))]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
bars_r2 = ax1.bar(fold_labels, cv_r2, color=C['verde'], alpha=0.8)
ax1.axhline(cv_r2.mean(), color=C['rojo'], lw=2, ls='--',
            label=f'Media: {cv_r2.mean():.3f}')
ax1.axhline(0, color='grey', lw=0.8)
for b, v in zip(bars_r2, cv_r2):
    ax1.text(b.get_x() + b.get_width() / 2,
             (v + 0.02 if v >= 0 else v - 0.06),
             f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')
ax1.set_ylabel('R²')
ax1.set_title('a) R² por fold walk-forward')
ax1.legend(fontsize=9)

ax2 = axes[1]
bars_rmse = ax2.bar(fold_labels, cv_rmse, color=C['naranja'], alpha=0.8)
ax2.axhline(cv_rmse.mean(), color=C['rojo'], lw=2, ls='--',
            label=f'Media: {cv_rmse.mean():.1f}')
for b, v in zip(bars_rmse, cv_rmse):
    ax2.text(b.get_x() + b.get_width() / 2, v + cv_rmse.max() * 0.02,
             f'{v:.0f}', ha='center', fontsize=8.5, fontweight='bold')
ax2.set_ylabel('RMSE')
ax2.set_title('b) RMSE por fold')
ax2.legend(fontsize=9)

plt.suptitle('Ilustración C6. Walk-Forward CV — Modelo C (XGBoost)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_06_cv_folds.png')
plt.close()

# ── Fig C-07: PDP univariados ─────────────────────────────────────────────────
print("  Fig C-07 — PDP")
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes      = axes.flatten()
pdp_colors = [C['azul'], C['naranja'], C['verde'], C['rojo'], C['purpura'], C['oscuro']]

for idx, (col, color) in enumerate(zip(X_COLS, pdp_colors)):
    ax = axes[idx]
    i  = X_COLS.index(col)
    if col == 'Dummy_COVID':
        means = []
        for v in [0, 1]:
            Xt = X_train.copy()
            Xt[:, i] = v
            means.append(xgb.predict(Xt).mean())
        ax.bar(['No COVID\n(0)', 'COVID\n(1)'], means,
               color=[C['azul'], C['rojo']], alpha=0.8, width=0.4)
        for j, val in enumerate(means):
            ax.text(j, val + abs(means[1]-means[0])*0.05,
                    f'{val:.0f}', ha='center', fontsize=11, fontweight='bold')
    else:
        grid = np.linspace(X_train[:, i].min(), X_train[:, i].max(), 50)
        pdp  = []
        for g in grid:
            Xt = X_train.copy()
            Xt[:, i] = g
            pdp.append(xgb.predict(Xt).mean())
        ax.plot(grid, pdp, color=color, lw=2.5)
        ax.fill_between(grid, min(pdp), pdp, alpha=0.08, color=color)
        ax.scatter(X_train[:, i], [min(pdp)] * len(X_train),
                   alpha=0.03, s=3, color='black', marker='|')
    ax.set_xlabel(LABEL.get(col, col), fontsize=9)
    ax.set_ylabel('Pred. media', fontsize=9)
    ax.set_title(col, fontsize=10)

plt.suptitle(
    'Ilustración C7. Dependencia Parcial (PDP) — Modelo C (XGBoost, 6 variables)\n'
    'Muestra el efecto marginal de cada variable manteniendo el resto constante',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_07_pdp.png')
plt.close()

# ── Fig C-08: PDP 2D o SHAP Dependence ───────────────────────────────────────
print("  Fig C-08 — PDP 2D / SHAP Dependence")
paro_idx  = X_COLS.index('Tasa_Paro')
renta_idx = X_COLS.index('Renta_Neta_Media')

if USE_SHAP:
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.dependence_plot(
        paro_idx, shap_values, X_test,
        feature_names=[LABEL.get(v, v) for v in X_COLS],
        interaction_index=renta_idx,
        show=False, ax=ax
    )
    ax.set_title(
        'Ilustración C8. SHAP Dependence — Tasa de Paro × Renta Neta Media\n'
        'Muestra la interacción no lineal capturada por XGBoost',
        fontsize=12, fontweight='bold', pad=15
    )
else:
    paro_grid  = np.linspace(X_train[:, paro_idx].min(),
                              X_train[:, paro_idx].max(), 40)
    renta_grid = np.linspace(X_train[:, renta_idx].min(),
                              X_train[:, renta_idx].max(), 40)
    pdp_2d = np.zeros((len(paro_grid), len(renta_grid)))
    for i, pv in enumerate(paro_grid):
        for j, rv in enumerate(renta_grid):
            Xt = X_train.copy()
            Xt[:, paro_idx]  = pv
            Xt[:, renta_idx] = rv
            pdp_2d[i, j] = xgb.predict(Xt).mean()

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.contourf(renta_grid, paro_grid, pdp_2d, levels=20,
                     cmap='RdYlGn_r', alpha=0.9)
    plt.colorbar(im, ax=ax, label='Predicción media (tasa delitos / 100k)')
    ax.scatter(X_test[:, renta_idx], X_test[:, paro_idx],
               c=y_test, cmap='RdYlGn_r', s=25,
               edgecolors='black', linewidths=0.5, alpha=0.8, zorder=5)
    ax.set_xlabel('Renta Neta Media (€)', fontsize=11)
    ax.set_ylabel('Tasa de Paro (%)', fontsize=11)
    ax.set_title(
        'Ilustración C8. PDP 2D — Tasa de Paro × Renta Neta Media (XGBoost)\n'
        'Rojo = mayor criminalidad predicha  |  Verde = menor criminalidad predicha',
        fontsize=12, pad=10
    )

plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_08_pdp2d.png')
plt.close()

# ── Fig C-09: Historial de optimización ──────────────────────────────────────
print("  Fig C-09 — Historial optimización")
df_trials = pd.DataFrame(trials_history)
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(df_trials['trial'], df_trials['cv_r2'],
           alpha=0.4, s=18, color=C['azul'], label='Trials')
best_so_far = df_trials['cv_r2'].cummax()
ax.plot(df_trials['trial'], best_so_far, color=C['rojo'], lw=2,
        label='Mejor acumulado')
ax.axhline(best_cv_r2, color=C['verde'], ls='--', lw=1.5, alpha=0.7,
           label=f'Mejor final: {best_cv_r2:.4f}')
ax.set_xlabel('Nº de trial')
ax.set_ylabel('CV-R² (Walk-Forward)')
ax.set_title(
    f'Ilustración C9. Historial de optimización de hiperparámetros\n'
    f'({N_TRIALS} trials — método: {"Optuna TPE Bayesiana" if USE_OPTUNA else "RandomSearch"})',
    fontsize=12, pad=10
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_09_optim.png')
plt.close()

# ── Fig C-10: Comparación Modelo B vs. Modelo C en test ──────────────────────
# Reentrenamos Modelo B (lineal con lag) sobre los mismos datos para comparar
print("  Fig C-10 — Comparación B vs. C en test")
scaler_B    = StandardScaler()
Xtr_B_sc    = scaler_B.fit_transform(X_train)
Xte_B_sc    = scaler_B.transform(X_test)
lr_B        = LinearRegression().fit(Xtr_B_sc, y_train)
y_pred_lr_B = lr_B.predict(Xte_B_sc)
m_B         = calc_metricas(y_test, y_pred_lr_B)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_pairs = [
    ('RMSE', m_B['RMSE'], m_te['RMSE'], '↓ mejor'),
    ('MAE',  m_B['MAE'],  m_te['MAE'],  '↓ mejor'),
    ('R²',   m_B['R2'],   m_te['R2'],   '↑ mejor'),
]
for ax, (metric, val_B, val_C, direction) in zip(axes, metric_pairs):
    colors = []
    for v_b, v_c in [(val_B, val_C)]:
        if metric == 'R²':
            colors = [C['azul'] if v_b >= v_c else C['gris'],
                      C['verde'] if v_c >= v_b else C['gris']]
        else:
            colors = [C['azul'] if v_b <= v_c else C['gris'],
                      C['verde'] if v_c <= v_b else C['gris']]
    bars = ax.bar(['Modelo B\n(Lineal+lag)', 'Modelo C\n(XGBoost)'],
                  [val_B, val_C], color=colors, alpha=0.85, width=0.45)
    for b, v in zip(bars, [val_B, val_C]):
        fmt = f'{v:.4f}' if metric == 'R²' else f'{v:.0f}'
        ax.text(b.get_x() + b.get_width() / 2,
                max(0, b.get_height()) * 1.02,
                fmt, ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} ({direction})', fontsize=11)
    if min(val_B, val_C) < 0:
        ax.set_ylim(min(val_B, val_C) * 1.4, max(val_B, val_C) * 1.25)

plt.suptitle(
    'Ilustración C10. Comparación Modelo B (Lineal) vs. Modelo C (XGBoost) — Test 2022-2024\n'
    'Azul = ganador en esa métrica',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_C_10_comparacion_BC.png')
plt.close()

# ==============================================================================
# 8. RESUMEN FINAL
# ==============================================================================
print(f"\n{'='*72}")
print("  RESUMEN FINAL — MODELO C (PREDICTIVO XGBOOST)")
print("="*72)
print(f"""
  Test:  R²={m_te['R2']:.4f}  RMSE={m_te['RMSE']:.0f}  MAE={m_te['MAE']:.0f}  MAPE={m_te['MAPE']:.1f}%
  Train: R²={m_tr['R2']:.4f}  RMSE={m_tr['RMSE']:.0f}  MAE={m_tr['MAE']:.0f}  MAPE={m_tr['MAPE']:.1f}%
  WF-CV: R²={cv_r2.mean():.4f} ± {cv_r2.std():.4f}
  Sobreajuste: {overfit:.4f} {'⚠ revisar' if overfit > 0.15 else '✓ aceptable'}
  Optimización: {"Optuna TPE" if USE_OPTUNA else "RandomSearch"} ({N_TRIALS} trials)
  Interpretabilidad: {"SHAP" if USE_SHAP else "Permutation Importance + PDP 2D"}

  Feature Importance (MDI):""")
for _, r in fi.iterrows():
    print(f"    {LABEL[r['Variable']]:35s}: {r['Importancia']*100:.1f}%")

print(f"""
  Comparación vs. Modelo B (mismo dataset):
    RMSE  — B: {m_B['RMSE']:.1f}  |  C: {m_te['RMSE']:.1f}  → ganador: {'B' if m_B['RMSE'] < m_te['RMSE'] else 'C'}
    R²    — B: {m_B['R2']:.4f}  |  C: {m_te['R2']:.4f}  → ganador: {'B' if m_B['R2'] > m_te['R2'] else 'C'}
    MAPE  — B: {m_B['MAPE']:.1f}%  |  C: {m_te['MAPE']:.1f}%  → ganador: {'B' if m_B['MAPE'] < m_te['MAPE'] else 'C'}

  Figuras generadas en {OUT}/:""")
for f in sorted(os.listdir(OUT)):
    print(f"    {f}")

[✓] XGBoost nativo detectado
[!] Optuna no disponible → RandomSearch como fallback
[✓] SHAP detectado

  MODELO C — PREDICTIVO AVANZADO (XGBoost con lag-1)

Dataset original: 780 filas × 17 columnas
Filas eliminadas (lag-1): 52
Dataset resultante: 728 filas (2011-2024)

Partición temporal:
  Train: 572 obs  (2011-2021)
  Test:  156 obs  (2022-2024)

Walk-Forward CV: 7 folds generados
  Fold 1: train 2011-2014 → val 2015
  Fold 2: train 2011-2015 → val 2016
  Fold 3: train 2011-2016 → val 2017
  Fold 4: train 2011-2017 → val 2018
  Fold 5: train 2011-2018 → val 2019
  Fold 6: train 2011-2019 → val 2020
  Fold 7: train 2011-2020 → val 2021

────────────────────────────────────────────────────────────────────────
  OPTIMIZACIÓN DE HIPERPARÁMETROS
────────────────────────────────────────────────────────────────────────
  Método: RandomSearch (50 iteraciones) + Walk-Forward manual
  Completado: 50 trials en 182.2s
  Mejor CV-R²: 0.6355

  Mejores hiperparámetros:
    n_estimators: 120
    m

In [4]:
!zip -r figuras_descarga.zip /content/figuras_modeloA/
!zip -r figuras_descarga.zip /content/figuras_modeloB/
!zip -r figuras_descarga.zip /content/figuras_modeloC/


  adding: content/figuras_modeloA/ (stored 0%)
  adding: content/figuras_modeloA/fig_A_04_evolucion.png (deflated 13%)
  adding: content/figuras_modeloA/fig_A_01_tabla_metricas.png (deflated 16%)
  adding: content/figuras_modeloA/fig_A_05_residuos.png (deflated 15%)
  adding: content/figuras_modeloA/fig_A_06_cv_folds.png (deflated 19%)
  adding: content/figuras_modeloA/fig_A_03_coeficientes.png (deflated 16%)
  adding: content/figuras_modeloA/fig_A_02_scatter.png (deflated 11%)
  adding: content/figuras_modeloA/fig_A_07_pdp.png (deflated 12%)
  adding: content/figuras_modeloB/ (stored 0%)
  adding: content/figuras_modeloB/fig_B_06_cv_folds.png (deflated 19%)
  adding: content/figuras_modeloB/fig_B_05_residuos.png (deflated 16%)
  adding: content/figuras_modeloB/fig_B_03_coeficientes.png (deflated 21%)
  adding: content/figuras_modeloB/fig_B_01_tabla_metricas.png (deflated 19%)
  adding: content/figuras_modeloB/fig_B_07_comparacion_coefs.png (deflated 18%)
  adding: content/figuras_mode

In [5]:
from google.colab import files
import os

# 1. Comprimimos las tres carpetas específicas en un solo archivo
!zip -r todas_las_figuras.zip figuras_modeloA figuras_modeloB figuras_modeloC

# 2. Forzamos la descarga al navegador
if os.path.exists('todas_las_figuras.zip'):
    files.download('todas_las_figuras.zip')
else:
    print("Error: No se ha podido crear el archivo zip. Revisa si estás en el directorio correcto.")

  adding: figuras_modeloA/ (stored 0%)
  adding: figuras_modeloA/fig_A_04_evolucion.png (deflated 13%)
  adding: figuras_modeloA/fig_A_01_tabla_metricas.png (deflated 16%)
  adding: figuras_modeloA/fig_A_05_residuos.png (deflated 15%)
  adding: figuras_modeloA/fig_A_06_cv_folds.png (deflated 19%)
  adding: figuras_modeloA/fig_A_03_coeficientes.png (deflated 16%)
  adding: figuras_modeloA/fig_A_02_scatter.png (deflated 11%)
  adding: figuras_modeloA/fig_A_07_pdp.png (deflated 12%)
  adding: figuras_modeloB/ (stored 0%)
  adding: figuras_modeloB/fig_B_06_cv_folds.png (deflated 19%)
  adding: figuras_modeloB/fig_B_05_residuos.png (deflated 16%)
  adding: figuras_modeloB/fig_B_03_coeficientes.png (deflated 21%)
  adding: figuras_modeloB/fig_B_01_tabla_metricas.png (deflated 19%)
  adding: figuras_modeloB/fig_B_07_comparacion_coefs.png (deflated 18%)
  adding: figuras_modeloB/fig_B_02_scatter.png (deflated 11%)
  adding: figuras_modeloB/fig_B_04_evolucion.png (deflated 13%)
  adding: figura

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>